In [1]:
import sqlite3
import pandas as pd 

In [2]:
conn = sqlite3.connect('data/youtube_data.db')

In [3]:
videos = pd.read_sql_query(
    '''
    select * from videos
    '''
, conn)
channels = pd.read_sql_query(
    '''
    select * from channels
    '''
, conn)


In [4]:
video_master = pd.read_sql_query('''
    select 
        v.video_id, v.channel_id, c.channel_title,
        v.title, v.published_at, v.duration_min,
        v.views, v.likes, v.comments_count,
        c.subscribers, c.country
    from videos v
    left join channels c on v.channel_id = c.channel_id
''', conn)

video_master

,video_id,channel_id,channel_title,title,published_at,duration_min,views,likes,comments_count,subscribers,country
0,eAYRGgocBSo,UCttLEDTJoFAwH6eGUKT7ilA,Traveler Ram,₹20 ka तीन 😂😂 #train #travel #shorts,2023-05-05T02:35:00Z,1.00,161570836,5677553,5869,2200000,IN
1,dO7VcTrZKjU,UCtfXgNnA-QcxjHJjk5wXLFg,Karl Rock,Shops with NO Shopkeepers!,2023-01-22T09:00:19Z,0.43,105530445,3657321,19398,3190000,IN
2,zsYDWBsLukQ,UCGeGhS_akOxBWQcSmje6B-w,Tanya Khanijow,Travelling on your Period can feel like this -...,2023-06-30T11:31:32Z,0.10,54376496,476196,5497,2230000,IN
3,aw90B5r7QaQ,UCd_vq7geqWx4if8HGJp7HkA,ABHINAV VLOG,Ladakh ride ep:-1 Follow for regular updates#l...,2023-06-17T16:05:54Z,1.02,48950174,2703099,2371,290000,IN
4,URnhIjHygrY,UCXzOPXsOLCJQcy1LUtUaUdg,The Indian Mukbanger,Experiencing 24 Hours in Rajdhani Express🚉😍,2024-10-03T06:30:14Z,0.95,59083669,2091349,1336,4300000,IN
...,...,...,...,...,...,...,...,...,...,...,...
713,UId8bYCP6Z4,UCFs61veTvwX7lzH4epq65xA,Travelling Knowledge,July mein ghoomne ka plan bana rahe ho #travel...,2026-07-02T15:17:28Z,1.17,1068,53,1,3520,IN
714,8fDe_dLuF4s,UCNWu-5luRb4YvUl27_Q9dxA,Hira Nag paglu uronchandi,"Dalhousie, Himachal Pradesh India guide, infor...",2026-07-02T15:30:19Z,0.32,54,1,2,314,IN
715,B0Mh1C8M3vI,UCyWyZZ_AnNmH8QGzUoUQ5uQ,Indian Travel Tour,Mountains Are Calling😍 #manali #travel #fyp #...,2026-06-30T13:50:57Z,0.48,913,45,1,400,IN
716,yxQqpK2w1Vw,UCv-i8_xm94fDgWaYHQJPXzw,Tourist Shubham,Why Foreigners LOVE India’s Golden Triangle! #...,2026-07-01T07:30:39Z,0.98,831,13,0,22300,IN


In [5]:
channel_summary = pd.read_sql_query('''
    WITH VideoAgg AS (
        SELECT 
            channel_id,
            COUNT(video_id) AS videos_in_dataset,
            SUM(views) AS total_views_dataset,
            AVG(views) AS avg_views,
            SUM(likes) AS total_likes,
            AVG(likes) AS avg_likes,
            SUM(comments_count) AS total_comments,
            AVG(comments_count) AS avg_comments,
            AVG(duration_min) AS avg_duration_min,
            SUM(CASE WHEN duration_min * 60 <= 60 THEN 1 ELSE 0 END) AS shorts_count
        FROM videos
        GROUP BY channel_id
    ),
    
    EngagementAgg AS (
        SELECT 
            channel_id,
            SUM(likes + comments_count) AS total_engagement,
            SUM(views) AS total_views_check
        FROM videos
        GROUP BY channel_id
    ),
    
    ChannelInfo AS (
        SELECT
            channel_id,
            channel_title,
            subscribers,
            total_videos AS channel_total_videos,
            total_views AS channel_total_views,
            country,
            created_date
        FROM channels
    )
    
    SELECT 
        ci.channel_id,
        ci.channel_title,
        ci.subscribers,
        ci.channel_total_videos,
        ci.channel_total_views,
        ci.country,
        ci.created_date,
        va.videos_in_dataset,
        va.total_views_dataset,
        va.avg_views,
        va.total_likes,
        va.avg_likes,
        va.total_comments,
        va.avg_comments,
        va.avg_duration_min,
        va.shorts_count,
        ea.total_engagement,
        ROUND(CAST(ea.total_engagement AS FLOAT) / NULLIF(ea.total_views_check, 0), 4) AS engagement_rate
    FROM ChannelInfo ci
    LEFT JOIN VideoAgg va ON ci.channel_id = va.channel_id
    LEFT JOIN EngagementAgg ea ON ci.channel_id = ea.channel_id
    ORDER BY va.total_views_dataset DESC
''', conn)

channel_summary

,channel_id,channel_title,subscribers,channel_total_videos,channel_total_views,country,created_date,videos_in_dataset,total_views_dataset,avg_views,total_likes,avg_likes,total_comments,avg_comments,avg_duration_min,shorts_count,total_engagement,engagement_rate
0,UCttLEDTJoFAwH6eGUKT7ilA,Traveler Ram,2200000,985,1168556943,IN,2021-11-21T09:18:13.841784Z,5,216810261,4.336205e+07,8488596,1.697719e+06,9631,1926.200000,0.934000,5,8498227,0.0392
1,UCgiWxZL6x7PrezXK5RuIxkw,therainbowgirl,4580000,1263,2752919063,IN,2021-05-17T13:07:14.910565Z,12,210553691,1.754614e+07,7688886,6.407405e+05,16645,1387.083333,0.905000,11,7705531,0.0366
2,UCtgGOdTlM-NdJ9rPKIYN8UQ,Slayy Point,10600000,198,3184653291,IN,2016-03-07T14:34:01Z,9,205834551,2.287051e+07,4322491,4.802768e+05,116825,12980.555556,24.586667,0,4439316,0.0216
3,UCFcBAGR0drI3F15EhMFjuTg,Aayu and Pihu Show,20600000,862,14481275330,IN,2017-05-08T19:27:48Z,16,186410799,1.165067e+07,1448697,9.054356e+04,139474,8717.125000,21.191250,0,1588171,0.0085
4,UCeMepUlAYyqintZfnxneFXg,Rational Society,964000,808,439171765,IN,2018-08-07T14:01:56Z,2,174136226,8.706811e+07,7734319,3.867160e+06,21323,10661.500000,0.760000,2,7755642,0.0445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,UCtZhguMdFX9IH-H07JbCSmA,Wanderlust and City Dust,5,1,329,IN,2009-11-04T18:11:57Z,1,329,3.290000e+02,1,1.000000e+00,0,0.000000,7.400000,0,1,0.0030
415,UCDC8u3-quEotNYPyBuJwCjw,The World Packer,47,54,13004,IN,2025-03-30T03:36:06.497114Z,1,251,2.510000e+02,4,4.000000e+00,4,4.000000,21.350000,0,8,0.0319
416,UCNWu-5luRb4YvUl27_Q9dxA,Hira Nag paglu uronchandi,314,287,172428,IN,2025-03-31T07:19:56.82353Z,1,54,5.400000e+01,1,1.000000e+00,2,2.000000,0.320000,1,3,0.0556
417,UC6gA-hutZnOcWoPesMX3Y6Q,Smart Travel Planner,8,49,2802,IN,2026-06-14T12:03:40.263998Z,1,53,5.300000e+01,2,2.000000e+00,1,1.000000,1.730000,0,3,0.0566


In [6]:
# Convert to datetime and shift to IST (audience is India-based)
video_master['published_at'] = pd.to_datetime(video_master['published_at'])
video_master['published_at_ist'] = video_master['published_at'].dt.tz_convert('Asia/Kolkata')

video_master['publish_weekday'] = video_master['published_at_ist'].dt.day_name()
video_master['publish_hour'] = video_master['published_at_ist'].dt.hour

def categorize_time(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

video_master['publish_time_category'] = video_master['publish_hour'].apply(categorize_time)

video_master['duration_sec'] = video_master['duration_min'] * 60
video_master['is_short'] = video_master['duration_sec'] <= 60

video_master['engagement_rate%'] = (
    ((video_master['likes'] + video_master['comments_count']) / video_master['views'])*100
).replace([float('inf'), -float('inf')], None)

video_master['title_length'] = video_master['title'].str.len()
video_master['title_word_count'] = video_master['title'].str.split().str.len()

keyword_flags = ['budget', 'solo', 'backpacking', 'hidden', 'guide', 'tips', 'best']
for kw in keyword_flags:
    video_master[f'title_has_{kw}'] = video_master['title'].str.lower().str.contains(kw)

In [7]:
channel_summary['engagement_rate%'] = (
    ((channel_summary['total_likes'] + channel_summary['total_comments']) 
    / channel_summary['total_views_dataset'])*100
)
channel_summary['created_date'] = pd.to_datetime(channel_summary['created_date'], format='ISO8601')

channel_summary['channel_age_months'] = (
    (pd.Timestamp.now(tz='UTC') - channel_summary['created_date']).dt.days / 30
)
channel_summary['upload_frequency_per_month'] = (
    channel_summary['channel_total_videos'] / channel_summary['channel_age_months']
)
def tier_channel(subs):
    if subs < 10000:
        return 'Small'
    elif subs < 100000:
        return 'Mid'
    else:
        return 'Large'

channel_summary['channel_tier'] = channel_summary['subscribers'].apply(tier_channel)

channel_summary['engagement_rate%'] = channel_summary['engagement_rate']*100
channel_summary= channel_summary.drop(columns='engagement_rate')

In [8]:
channel_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 419 entries, 0 to 418
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   channel_id                  419 non-null    object             
 1   channel_title               419 non-null    object             
 2   subscribers                 419 non-null    int64              
 3   channel_total_videos        419 non-null    int64              
 4   channel_total_views         419 non-null    int64              
 5   country                     419 non-null    object             
 6   created_date                419 non-null    datetime64[ns, UTC]
 7   videos_in_dataset           419 non-null    int64              
 8   total_views_dataset         419 non-null    int64              
 9   avg_views                   419 non-null    float64            
 10  total_likes                 419 non-null    int64             

In [9]:
channel_summary.describe()

,subscribers,channel_total_videos,channel_total_views,videos_in_dataset,total_views_dataset,avg_views,total_likes,avg_likes,total_comments,avg_comments,avg_duration_min,shorts_count,total_engagement,engagement_rate%,channel_age_months,upload_frequency_per_month
count,4.190000e+02,419.000000,4.190000e+02,419.000000,4.190000e+02,4.190000e+02,4.190000e+02,4.190000e+02,419.000000,419.000000,419.000000,419.000000,4.190000e+02,419.000000,419.000000,419.000000
mean,1.941466e+06,833.159905,1.072540e+09,1.713604,1.010268e+07,5.055615e+06,2.848531e+05,1.367464e+05,3214.749403,1425.441613,9.610657,0.830549,2.880679e+05,2.548210,90.329833,10.382551
std,6.209622e+06,2987.759099,4.073030e+09,2.393988,2.816190e+07,1.323903e+07,9.307170e+05,3.854812e+05,11864.880039,4479.075983,13.824280,1.859264,9.357918e+05,2.959397,50.116307,24.164997
min,5.000000e+00,1.000000,3.290000e+02,1.000000,1.000000e+01,1.000000e+01,0.000000e+00,0.000000e+00,0.000000,0.000000,0.080000,0.000000,1.000000e+00,0.000000,1.400000,0.004905
25%,1.675000e+04,186.000000,7.140006e+06,1.000000,4.406095e+05,4.359490e+05,4.676500e+03,4.327000e+03,91.500000,85.750000,0.650000,0.000000,4.904500e+03,1.175000,55.150000,2.681679
50%,1.670000e+05,421.000000,5.621714e+07,1.000000,1.650662e+06,1.290338e+06,2.624400e+04,2.057100e+04,401.000000,326.666667,1.020000,1.000000,2.635600e+04,2.070000,80.200000,5.386563
75%,1.135000e+06,819.500000,4.348658e+08,1.000000,6.343318e+06,4.110792e+06,1.346535e+05,9.243050e+04,1547.000000,1053.000000,15.740000,1.000000,1.373100e+05,3.245000,121.583333,10.901098
max,5.200000e+07,58216.000000,4.507125e+10,31.000000,2.168103e+08,1.569859e+08,8.488596e+06,3.867160e+06,139474.000000,73244.000000,91.550000,31.000000,8.498227e+06,50.000000,234.100000,390.798836


In [10]:
channel_summary.isnull().sum()

channel_id                    0
channel_title                 0
subscribers                   0
channel_total_videos          0
channel_total_views           0
country                       0
created_date                  0
videos_in_dataset             0
total_views_dataset           0
avg_views                     0
total_likes                   0
avg_likes                     0
total_comments                0
avg_comments                  0
avg_duration_min              0
shorts_count                  0
total_engagement              0
engagement_rate%              0
channel_age_months            0
upload_frequency_per_month    0
channel_tier                  0
dtype: int64

In [11]:
video_master.sample()

,video_id,channel_id,channel_title,title,published_at,duration_min,views,likes,comments_count,subscribers,...,engagement_rate%,title_length,title_word_count,title_has_budget,title_has_solo,title_has_backpacking,title_has_hidden,title_has_guide,title_has_tips,title_has_best
253,pAqJ6xkfKBQ,UCeMepUlAYyqintZfnxneFXg,Rational Society,Female Travel Without Money,2022-12-17 17:21:55+00:00,0.65,78962815,3666195,11688,964000,...,4.65774,27,4,False,False,False,False,False,False,False


In [13]:
video_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718 entries, 0 to 717
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype                       
---  ------                 --------------  -----                       
 0   video_id               718 non-null    object                      
 1   channel_id             718 non-null    object                      
 2   channel_title          718 non-null    object                      
 3   title                  718 non-null    object                      
 4   published_at           718 non-null    datetime64[ns, UTC]         
 5   duration_min           718 non-null    float64                     
 6   views                  718 non-null    int64                       
 7   likes                  718 non-null    int64                       
 8   comments_count         718 non-null    int64                       
 9   subscribers            718 non-null    int64                       
 10  country       

In [14]:
video_master.describe()

,duration_min,views,likes,comments_count,subscribers,publish_hour,duration_sec,engagement_rate%,title_length,title_word_count
count,718.000000,7.180000e+02,7.180000e+02,718.000000,7.180000e+02,718.000000,718.000000,718.000000,718.000000,718.000000
mean,10.213217,5.895576e+06,1.662304e+05,1876.016713,2.663099e+06,15.281337,612.793036,2.613263,72.739554,11.912256
std,13.851035,1.394341e+07,4.518915e+05,4346.222555,6.697395e+06,4.611121,831.062120,2.567344,22.411527,3.993278
min,0.080000,1.000000e+01,0.000000e+00,0.000000,5.000000e+00,0.000000,4.800000,0.000000,12.000000,2.000000
25%,0.680000,6.070042e+05,7.398750e+03,108.750000,5.095000e+04,11.000000,40.800000,1.280245,56.000000,9.000000
50%,1.020000,1.781171e+06,3.352400e+04,514.000000,3.930000e+05,16.000000,61.200000,2.118752,77.000000,12.000000
75%,18.072500,5.296537e+06,1.277512e+05,1535.500000,2.200000e+06,19.000000,1084.350000,3.405366,93.000000,15.000000
max,91.550000,1.615708e+08,5.677553e+06,73244.000000,5.200000e+07,23.000000,5493.000000,50.000000,100.000000,22.000000


In [15]:
video_master.to_sql('video_master', conn, if_exists='replace', index=False)
channel_summary.to_sql('channel_summary', conn, if_exists='replace', index=False)

print(f"video_master: {len(video_master)} rows")
print(f"channel_summary: {len(channel_summary)} rows")

video_master: 718 rows
channel_summary: 419 rows


In [18]:
video_master.to_csv("Video_Summary.csv", index=False)
channel_summary.to_csv("Channel_Summary.csv", index=False)